In [0]:
%run ../config_setup

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.getOrCreate()

# KPI: MAINTENANCE URGENCY SCORE (Combining Date + Telemetry)
# Logic: (Days since last maint * 0.3) + (Overheat count * 0.4) + (Idle ratio * 0.3)
telemetry_stats = spark.read.table(f"{GOLD}.fact_telemetry_30min").groupBy("machine_sk").agg(
    count(when(col("max_temp") > 120, 1)).alias("overheat_count")
)
last_maint = spark.read.table(f"{GOLD}.fact_maintenance_event").groupBy("machine_sk").agg(
    max("days_since_last_maintenance").alias("days_since_maint")
)
util_stats = spark.read.table(f"{GOLD}.fact_utilization_daily").groupBy("machine_sk").agg(
    avg("utilization_pct").alias("avg_util")
)

# 👉 FIX: fillna sirf measure columns pe — machine_sk (join key) pe nahi!
# (pehle whole-DF fillna(0) machine_sk ko bhi 0 kar deta tha → dim_machine join toot jata tha)
urgency = telemetry_stats.join(last_maint, "machine_sk", "outer").join(util_stats, "machine_sk", "outer") \
    .fillna(0, subset=["overheat_count", "days_since_maint", "avg_util"]) \
    .withColumn("urgency_score", 
                least(round((col("days_since_maint") * 0.3) + (col("overheat_count") * 0.4) + ((1 - col("avg_util")) * 30), 2), lit(100.0)))
urgency.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD}.kpi_maintenance_urgency")
print("✅ gold_08_kpi_predictive built (fillna subset fix ke saath).")